# LFM2.5-24B-A2B expansion eval (run 027)
Serves LiquidAI/LFM2.5-24B-A2B-GGUF Q6_K on CPU, runs the same 45-IR
expansion set with per-call timing capture, saves replies-lfm25-probe.jsonl.
Scoring happens LOCALLY. Settings: no accelerator, Internet ON.


In [ ]:
import subprocess
import sys


def pip_install(*args):
    print('+', 'pip install', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                     check=True)


pip_install('--upgrade', 'pip')
pip_install('llama-cpp-python')
# Server runtime closure (bare install lacks it; v3/v2 died on sse_starlette).
pip_install('llama-cpp-python[server]')
pip_install('sse-starlette', 'starlette-context', 'fastapi',
            'uvicorn[standard]', 'pydantic', 'pydantic-settings',
            'anyio', 'httpx')
pip_install('huggingface_hub', 'hf_transfer')
import llama_cpp
import sse_starlette, starlette_context, fastapi, uvicorn  # noqa: F401
from llama_cpp.server.app import create_app  # noqa: F401
print('server closure OK (import chain verified pre-download)')
print('llama-cpp-python OK (CPU build)')


In [ ]:
import json as _json
import sys as _sys
import urllib.request as _urlreq

TUNNEL = ""  # stamped by phonehome/push.sh (public receiver URL); empty = disabled
TOKEN = ""  # stamped by phonehome/push.sh (bearer token); empty = disabled
KID = 'cpu-lfm25-probe'  # per-kernel short id (cpu-shootout, cpu-lfm25, cpu-lfm25-probe, ...)


def hb(phase, msg=""):
    if not TUNNEL or not TOKEN:
        return
    try:
        body = _json.dumps(
            {"kernel": KID, "phase": phase, "msg": str(msg)[:300]}
        ).encode()
        req = _urlreq.Request(
            f"{TUNNEL}/hb?token={TOKEN}",
            data=body,
            headers={"Content-Type": "application/json"},
        )
        with _urlreq.urlopen(req, timeout=5):
            pass
    except Exception:
        pass


_prev_hook = _sys.excepthook


def _hook(t, v, tb, _p=_prev_hook):
    hb("fatal", f"{t.__name__}: {v}")
    return _p(t, v, tb)


_sys.excepthook = _hook
print("phone-home:", "ON" if TUNNEL and TOKEN else "OFF (unstamped)")

import json
import os
import subprocess
import sys
import time
import urllib.request

REPO = 'LiquidAI/LFM2.5-24B-A2B-GGUF'
GGUF_FILE = 'LFM2.5-24B-A2B-Q6_K.gguf'
N_CTX = 8192
PORT = 8083
TEMP = 0
MAX_TOKENS = 512

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

D = json.load(open('/kaggle/input/datasets/blonded2/qwen-eval-irs/irs.json'))
P = json.load(open('/kaggle/input/datasets/blonded2/qwen-eval-irs/prompts.json'))
assert len(D) == 45, len(D)
PROMPT = P['v0']
assert '{draft}' in PROMPT and 'TESTDRAFT' not in PROMPT
print('eval set OK:', len(D), 'IRs, single arm')
hb('eval-ok', f'{len(D)} IRs')

hb('downloading', GGUF_FILE)
path = hf_hub_download(repo_id=REPO, filename=GGUF_FILE)
print('downloaded:', path, flush=True)
hb('downloaded', GGUF_FILE)

log = open('/tmp/llama-server.log', 'w')
cmd = [sys.executable, '-m', 'llama_cpp.server',
       '--model', path, '--n_gpu_layers', '0',
       '--n_ctx', str(N_CTX), '--host', '127.0.0.1',
       '--port', str(PORT)]
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for i in range(60):
    if proc.poll() is not None:
        print('--- server died; log tail ---')
        print(''.join(open('/tmp/llama-server.log').readlines()[-40:]))
        print('--- env capsule ---')
        print(subprocess.run('free -g; df -h /tmp | tail -1', shell=True, capture_output=True, text=True).stdout)
        raise RuntimeError(f"server exited with code {proc.returncode}")
    time.sleep(10)
    try:
        with urllib.request.urlopen(
                f'http://127.0.0.1:{PORT}/v1/models', timeout=5) as r:
            print('READY, status', r.status, flush=True)
            break
            hb('ready', 'lfm25-probe')
    except Exception:
        if i % 6 == 0:
            print(f'waiting for server... ({i * 10}s)', flush=True)
else:
    raise RuntimeError('server did not come up in 10 min')

# --- technique matrix (LFM2.5 diagnosis probe; one variable per arm) ---
DEMO_IR = "Appreciate interest in sharing thoughts about service\nAssistance available to submit feedback"
DEMO_REPLY = "We appreciate your interest in sharing your thoughts about our service. Assistance is available to help you submit your feedback."
BLUNT = 'Output ONLY the customer reply. No analysis, no quoted rules, no extra text.\nDRAFT:\n{draft}\nREPLY:\n'

ARMS = [
    ('control', lambda ir: PROMPT.replace('{draft}', ir), 0, None, None, 512, D[0:2]),
    ('card-sample', lambda ir: PROMPT.replace('{draft}', ir), 0.1, 50, 1.05, 512, D[2:4]),
    ('cap1024', lambda ir: PROMPT.replace('{draft}', ir), 0, None, None, 1024, D[4:6]),
    ('oneshot', lambda ir: PROMPT.replace('{draft}', DEMO_IR + '\nREPLY:\n' + DEMO_REPLY + '\n\nDRAFT:\n' + ir), 0, None, None, 512, D[6:8]),
    ('blunt', lambda ir: BLUNT.replace('{draft}', ir), 0, None, None, 512, D[8:10]),
]


def gen(prompt, temp, top_k, rep_pen, max_tok):
    body = {'messages': [{'role': 'user', 'content': prompt}], 'temperature': temp, 'max_tokens': max_tok}
    if top_k is not None:
        body['top_k'] = top_k
    if rep_pen is not None:
        body['repeat_penalty'] = rep_pen
    data = json.dumps(body).encode()
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/chat/completions', data=data, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.load(r)


try:
    with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/props', timeout=15) as r:
        props = json.load(r)
    print('props keys:', sorted(props.keys()))
    print('chat_template:', str(props.get('chat_template'))[:600])
    hb('props', str(sorted(props.keys()))[:200])
except Exception as e:
    print('props fetch failed:', type(e).__name__, str(e)[:150])
try:
    tpl = subprocess.run(['grep', '-a', '-o', '-E', '.{0,40}tokenizer.chat_template.{0,300}', path], capture_output=True, text=True).stdout
    print('gguf template frag:', (tpl[:500] if tpl else 'NOT FOUND'))
except Exception as e2:
    print('template grep failed:', type(e2).__name__)

print('arm keys:', [(a[0], [i['key'] for i in a[6]]) for a in ARMS], flush=True)
outp = '/kaggle/working/replies-lfm25-probe.jsonl'
n_err = 0
with open(outp, 'w') as out:
    for name, build, temp, top_k, rep_pen, max_tok, items in ARMS:
        for item in items:
            prompt = build(item['ir'])
            row = {'key': item['key'], 'tech': name, 'reply': '', 'finish': '', 'wallMs': 0, 'compTokens': None, 'error': None}
            ok = False
            for attempt in range(2):
                try:
                    tr0 = time.time()
                    d = gen(prompt, temp, top_k, rep_pen, max_tok)
                    row['wallMs'] = round((time.time() - tr0) * 1000)
                    row['reply'] = d['choices'][0]['message']['content'] or ''
                    row['finish'] = d['choices'][0]['finish_reason']
                    usg = d.get('usage') or {}
                    row['compTokens'] = usg.get('completion_tokens')
                    ok = True
                    break
                except Exception as e:
                    row['error'] = f'{type(e).__name__}: {e}'[:200]
                    time.sleep(10)
            if not ok:
                n_err += 1
            out.write(json.dumps(row) + '\n')
            out.flush()
        hb('arm-done', f'{name} {n_err}err')
        print(f'arm {name} done, {n_err} errors', flush=True)
        assert n_err <= 4, f'error budget exceeded ({n_err})'
print('done')
hb('done', 'lfm25 probe complete')
